# Fine-tuning the ticket router

Three planted defects:

* cell 4 builds the training `DataLoader` with no `shuffle=` and no
  `sampler=`, so every epoch sees the tickets in filing order (MLV110);
* cell 4 builds the validation loader with `shuffle=True` (MLV111);
* cell 7 evaluates without `model.eval()` and without `torch.no_grad()`,
  so dropout stays on and the reported accuracy is wrong (MLV301,
  MLV302).

`execution_count` is monotonic, so document order is the run order.

In [1]:
%matplotlib inline
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(4242)
np.random.seed(4242)

In [2]:
tokens = torch.randint(0, 4096, (6000, 32))
labels = torch.randint(0, 7, (6000,))
cut = 4800
train_set = TensorDataset(tokens[:cut], labels[:cut])
val_set = TensorDataset(tokens[cut:], labels[cut:])
len(train_set), len(val_set)

In [3]:
class TicketRouter(nn.Module):
    def __init__(self, vocab=4096, width=128, classes=7):
        super().__init__()
        self.embed = nn.Embedding(vocab, width)
        self.body = nn.Sequential(
            nn.Linear(width, width),
            nn.GELU(),
            nn.Dropout(0.3),
        )
        self.head = nn.Linear(width, classes)

    def forward(self, x):
        return self.head(self.body(self.embed(x).mean(dim=1)))

model = TicketRouter()
model

In [4]:
train_loader = DataLoader(train_set, batch_size=64)
val_loader = DataLoader(val_set, batch_size=64, shuffle=True)

In [5]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

In [6]:
for epoch in range(5):
    model.train()
    running = 0.0
    for x, y in train_loader:
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        running += loss.item()
    print(epoch, running / len(train_loader))

In [7]:
correct = 0
seen = 0
for x, y in val_loader:
    predicted = model(x).argmax(dim=1)
    correct += (predicted == y).sum().item()
    seen += y.numel()
print("val accuracy", correct / seen)